# 🛒 Caso 1 — Loja Verde: clientes, compras y dashboard de ventas

> **Módulos del MBA relacionados:** Introdução à Programação com Python I–III · Business Intelligence e Data Visualization I–III

## El caso
La **Loja Verde** es un e-commerce brasileño de productos naturales y sustentables: cosméticos veganos, alimentos orgánicos, suplementos y artículos para una casa sustentable. Vende a todo Brasil desde 2023.

Marina, la fundadora, te contrató como analista de datos. Te mandó dos archivos exportados del sistema ("medio bagunçados", según ella) y te pidió un **dashboard para la reunión de directorio** de la semana que viene. Quiere decidir **dónde invertir en marketing en 2026**.

## Tu misión: responder estas preguntas
1. ¿Cuánto facturamos en 2024–2025 y cómo evoluciona mes a mes? ¿Crecimos?
2. ¿Cuál es el ticket medio (valor promedio por pedido)?
3. ¿Qué categorías y productos generan más ingresos?
4. ¿De qué estados vienen las ventas?
5. ¿Qué canal de adquisición trae los clientes **más valiosos**?
6. ¿Qué porcentaje de pedidos se cancela o se devuelve?
7. ¿Quiénes son los mejores clientes y quiénes están por irse? (segmentación **RFM**)

## Diccionario de datos
**clientes.csv** (separador `,`)

| columna | descripción |
|---|---|
| `cliente_id` | identificador único del cliente |
| `nome` | nombre |
| `data_cadastro` | fecha de registro (aaaa-mm-dd) |
| `cidade`, `estado` | ciudad y sigla del estado ⚠️ `estado` viene sucio (" sp", "Sp ") |
| `canal_aquisicao` | cómo llegó: Instagram, Google, TikTok, Indicação, E-mail |
| `idade` | edad ⚠️ hay vacíos |

**pedidos.csv** (separador `;` ⚠️)

| columna | descripción |
|---|---|
| `pedido_id` | identificador del pedido ⚠️ hay filas duplicadas |
| `cliente_id` | quién compró (la "llave" para unir con clientes) |
| `data_pedido` | fecha como texto dd/mm/aaaa ⚠️ |
| `produto`, `categoria` | qué compró |
| `quantidade` | unidades ⚠️ hay errores (0 o negativos) |
| `preco_unitario` | texto con formato "R\$ 1.234,56" ⚠️ |
| `status` | Entregue, Cancelado o Devolvido |
| `avaliacao` | nota de 1 a 5 (vacío si el cliente no evaluó) |

## Qué vas a practicar
`read_csv` con separador · `info` / `describe` / `isna` · limpieza de texto (`str.replace`, `str.strip`) · tipos y fechas (`astype`, `to_datetime`) · `drop_duplicates` · `merge` · `groupby` + `agg` · `crosstab` · `qcut` · funciones y `apply` · dashboard con Plotly · exportar a HTML

> **Cómo usar este notebook:** (1) ejecuta el Paso 0 para crear los archivos; (2) en la **Parte A** lee los comentarios de cada paso y escribe tu código; (3) después compara con la **Parte B** (solución).

## Paso 0 — Genera la base de datos

Este código **simula** las dos exportaciones del sistema de la Loja Verde y las guarda como `clientes.csv` y `pedidos.csv`
(en Colab aparecen en el panel 📁 de la izquierda). Ejecútalo una vez.

No necesitas entenderlo ahora. Cuando termines el curso de Python, vuelve y léelo: es un buen ejercicio de numpy.

In [ ]:
# ============================================================
# PASO 0 — GENERADOR DE DATOS (ejecútalo una vez)
# ============================================================
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)  # semilla fija → todas obtenemos exactamente los mismos datos

# ---------- 1) CLIENTES ----------
n_clientes = 1200
nomes = ["Ana", "Bruno", "Carla", "Diego", "Eduarda", "Felipe", "Gabriela", "Henrique",
         "Isabela", "João", "Larissa", "Marcos", "Natália", "Otávio", "Paula", "Rafael",
         "Sofia", "Thiago", "Vanessa", "Yuri"]
sobrenomes = ["Silva", "Santos", "Oliveira", "Souza", "Lima", "Pereira", "Costa",
              "Rodrigues", "Almeida", "Nascimento", "Ferreira", "Carvalho"]
cidades = pd.DataFrame({
    "cidade": ["São Paulo", "Campinas", "Piracicaba", "Rio de Janeiro", "Belo Horizonte",
               "Curitiba", "Porto Alegre", "Florianópolis", "Salvador", "Recife", "Brasília"],
    "estado": ["SP", "SP", "SP", "RJ", "MG", "PR", "RS", "SC", "BA", "PE", "DF"],
    "peso":   [30, 8, 5, 15, 10, 8, 7, 5, 5, 4, 3],
})
canais = ["Instagram", "Google", "TikTok", "Indicação", "E-mail"]
peso_canais = [0.35, 0.25, 0.15, 0.15, 0.10]
propensao = {"Instagram": 1.0, "Google": 0.9, "TikTok": 0.7, "Indicação": 1.7, "E-mail": 1.3}

idx_cidade = rng.choice(len(cidades), size=n_clientes, p=cidades["peso"] / cidades["peso"].sum())
data_cadastro = pd.Timestamp("2023-01-01") + pd.to_timedelta(rng.integers(0, 1004, n_clientes), unit="D")

clientes = pd.DataFrame({
    "cliente_id": np.arange(1, n_clientes + 1),
    "nome": [f"{rng.choice(nomes)} {rng.choice(sobrenomes)}" for _ in range(n_clientes)],
    "data_cadastro": data_cadastro.strftime("%Y-%m-%d"),
    "cidade": cidades["cidade"].to_numpy()[idx_cidade],
    "estado": cidades["estado"].to_numpy()[idx_cidade],
    "canal_aquisicao": rng.choice(canais, size=n_clientes, p=peso_canais),
    "idade": rng.normal(34, 10, n_clientes).clip(18, 75).round(),
})
# "Ensuciamos" los datos como en la vida real
sujo = rng.random(n_clientes) < 0.12
clientes.loc[sujo, "estado"] = [rng.choice([e.lower(), f" {e}", f"{e} ", e.title()])
                                for e in clientes.loc[sujo, "estado"]]
clientes.loc[rng.random(n_clientes) < 0.05, "idade"] = np.nan

# ---------- 2) PRODUCTOS ----------
produtos = pd.DataFrame([
    ("Shampoo sólido", "Cosméticos veganos", 39.90), ("Sérum vitamina C", "Cosméticos veganos", 89.90),
    ("Protetor solar mineral", "Cosméticos veganos", 69.90), ("Desodorante natural", "Cosméticos veganos", 34.90),
    ("Hidratante corporal", "Cosméticos veganos", 54.90), ("Café orgânico 500g", "Alimentos orgânicos", 42.90),
    ("Mel silvestre 500g", "Alimentos orgânicos", 36.90), ("Granola artesanal", "Alimentos orgânicos", 29.90),
    ("Azeite extravirgem 500ml", "Alimentos orgânicos", 64.90), ("Castanha-do-pará 200g", "Alimentos orgânicos", 32.90),
    ("Whey vegano 900g", "Suplementos", 159.90), ("Colágeno verisol", "Suplementos", 119.90),
    ("Ômega 3 vegano", "Suplementos", 99.90), ("Spirulina", "Suplementos", 74.90),
    ("Kit canudos inox", "Casa sustentável", 29.90), ("Escova de bambu", "Casa sustentável", 14.90),
    ("Sabão de coco", "Casa sustentável", 12.90), ("Bucha vegetal", "Casa sustentável", 9.90),
    ("Garrafa térmica", "Casa sustentável", 119.90), ("Ecobag", "Casa sustentável", 24.90),
], columns=["produto", "categoria", "preco_base"])
peso_produtos = np.array([6, 5, 5, 6, 5, 9, 7, 8, 5, 6, 4, 4, 3, 3, 4, 6, 7, 5, 3, 4], dtype=float)

# ---------- 3) PEDIDOS (con crecimiento, estacionalidad y clientes que dejan de comprar) ----------
dias = pd.date_range("2024-01-01", "2025-12-31", freq="D")
sazonal = np.select([dias.month == 5, dias.month == 11, dias.month == 12], [1.25, 1.7, 1.4], 1.0)
acum = np.cumsum((1 + 0.3 * np.arange(len(dias)) / len(dias)) * sazonal)

ini = (pd.to_datetime(clientes["data_cadastro"]) - dias[0]).dt.days.clip(lower=0).to_numpy()
churn = rng.random(n_clientes) < 0.45
fim = np.where(churn, np.minimum(ini + rng.integers(30, 450, n_clientes), len(dias) - 1), len(dias) - 1)
fator = rng.gamma(2.0, 0.5, n_clientes)
lam = clientes["canal_aquisicao"].map(propensao).to_numpy() * fator * (fim - ini + 1) / 365 * 4.5
cli = np.repeat(np.arange(n_clientes), rng.poisson(lam))

lo = np.where(ini[cli] > 0, acum[ini[cli] - 1], 0)
hi = acum[fim[cli]]
datas = dias[np.searchsorted(acum, lo + rng.random(len(cli)) * (hi - lo)).clip(max=len(dias) - 1)]

n = len(cli)
p_idx = rng.choice(len(produtos), size=n, p=peso_produtos / peso_produtos.sum())
preco = produtos["preco_base"].to_numpy()[p_idx]
preco = np.where(datas.month == 11, (preco * 0.85).round(2), preco)  # descuento Black Friday
status = rng.choice(["Entregue", "Cancelado", "Devolvido"], size=n, p=[0.9, 0.06, 0.04])
aval = rng.choice([1, 2, 3, 4, 5], size=n, p=[0.03, 0.05, 0.12, 0.35, 0.45]).astype(float)
aval[(status != "Entregue") | (rng.random(n) < 0.4)] = np.nan

pedidos = pd.DataFrame({
    "data": datas, "cliente_id": cli + 1,
    "produto": produtos["produto"].to_numpy()[p_idx], "categoria": produtos["categoria"].to_numpy()[p_idx],
    "quantidade": rng.choice([1, 2, 3, 4, 5], size=n, p=[0.68, 0.2, 0.07, 0.03, 0.02]),
    "preco": preco, "status": status, "avaliacao": aval,
}).sort_values("data", kind="stable").reset_index(drop=True)
pedidos.insert(0, "pedido_id", np.arange(100001, 100001 + n))

# Formato "brasileño" de fecha y dinero (texto), errores de digitación y duplicados
pedidos["data_pedido"] = pedidos["data"].dt.strftime("%d/%m/%Y")
pedidos["preco_unitario"] = ["R$ " + f"{v:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
                             for v in pedidos["preco"]]
erro = rng.choice(n, 25, replace=False)
pedidos.loc[erro, "quantidade"] = rng.choice([0, -1], 25)
pedidos = pd.concat([pedidos, pedidos.sample(60, random_state=1)]).sort_values("pedido_id", kind="stable")

colunas = ["pedido_id", "cliente_id", "data_pedido", "produto", "categoria",
           "quantidade", "preco_unitario", "status", "avaliacao"]
pedidos[colunas].to_csv("pedidos.csv", sep=";", index=False)
clientes.to_csv("clientes.csv", index=False)
print(f"✅ Archivos creados: clientes.csv ({len(clientes)} filas) y pedidos.csv ({len(pedidos)} filas)")

---
# ✍️ Parte A — Tu turno: el paso a paso en comentarios

Cada celda de abajo tiene **solo comentarios** con lo que debes hacer y algunas pistas.
Escribe tu código debajo de `# ✍️ Tu código aquí:` y ejecútalo.

- Si no conoces una función, búscala ("pandas como convertir texto a fecha"): buscar es parte del trabajo de un analista.
- Equivocarse es normal. Lee el mensaje de error de abajo hacia arriba: la última línea dice qué pasó.
- Si una celda que modifica datos se ejecuta dos veces, puede dar error. Vuelve a ejecutar desde el paso donde lees los archivos.

### Paso 1 — Importar las librerías

Toda análisis empieza cargando las herramientas: **pandas** (tablas), **numpy** (números) y **plotly** (gráficos interactivos).

In [ ]:
# PASO 1 — IMPORTAR LAS LIBRERÍAS
# 1. Importa pandas con el apodo pd y numpy con el apodo np
# 2. Importa plotly.express como px y plotly.graph_objects como go
# 3. Importa make_subplots desde plotly.subplots (lo usarás para armar el dashboard)
# 💡 Pista: import pandas as pd

# ✍️ Tu código aquí:


### Paso 2 — Importar las bases de datos

Lee los dos archivos CSV. Ojo: no todos los CSV usan coma como separador.

In [ ]:
# PASO 2 — IMPORTAR LAS BASES DE DATOS
# 1. Lee "clientes.csv" con pd.read_csv y guárdalo en una variable llamada clientes
# 2. Lee "pedidos.csv" en una variable llamada pedidos
#    ⚠️ Este archivo usa ";" como separador → usa el parámetro sep=";"
# 3. Muestra cuántas filas y columnas tiene cada tabla con .shape
# 4. Mira las primeras 5 filas de pedidos con .head()

# ✍️ Tu código aquí:


### Paso 3 — Explorar y diagnosticar los problemas

Antes de calcular nada, haz un **diagnóstico**: tipos de cada columna, vacíos, duplicados y valores raros.

In [ ]:
# PASO 3 — EXPLORAR Y DIAGNOSTICAR LOS PROBLEMAS
# 1. Usa pedidos.info() → ¿qué tipo tienen preco_unitario y data_pedido? (si dice object/str, es texto ❌)
# 2. Cuenta los vacíos por columna: pedidos.isna().sum()
# 3. Cuenta las filas duplicadas: pedidos.duplicated().sum()
# 4. Cuenta cuántas filas tienen quantidade <= 0
# 5. Mira los valores distintos de clientes["estado"] con .value_counts() → ¿ves "sp", " SP", "Sp"?
# 6. Usa clientes.describe() para ver min/máx/media de la edad

# ✍️ Tu código aquí:


### Paso 4 — Limpiar y ajustar el formato de los datos

Arregla cada problema del diagnóstico. Este paso suele llevar el **70% del tiempo** de un proyecto de datos.

In [ ]:
# PASO 4 — LIMPIAR Y AJUSTAR EL FORMATO DE LOS DATOS
# 4.1 Elimina las filas duplicadas de pedidos con .drop_duplicates() y muestra cuántas quitaste
# 4.2 Convierte preco_unitario de texto ("R$ 1.234,56") a número (1234.56):
#     - quita "R$" con .str.replace("R$", "", regex=False)
#     - quita el punto de los miles
#     - cambia la coma decimal por punto
#     - quita espacios con .str.strip() y convierte con .astype(float)
#     ⚠️ El ORDEN importa: si cambias la coma por punto antes de quitar los puntos, ¡arruinas el número!
# 4.3 Convierte data_pedido a fecha: pd.to_datetime(..., format="%d/%m/%Y")
#     y data_cadastro de clientes también (ese ya viene como aaaa-mm-dd)
# 4.4 Estandariza estado: quita espacios (.str.strip()) y pasa a mayúsculas (.str.upper())
# 4.5 Quédate solo con los pedidos con quantidade > 0
# 4.6 Rellena la edad vacía con la mediana: .fillna(clientes["idade"].median())
# 4.7 Verifica con .info() y .value_counts() que todo quedó bien

# ✍️ Tu código aquí:


### Paso 5 — Crear columnas nuevas

Crea las variables que vas a necesitar para analizar: el valor de cada pedido, el año y el mes.

In [ ]:
# PASO 5 — CREAR COLUMNAS NUEVAS
# 1. Crea valor_total = quantidade * preco_unitario
# 2. Crea la columna ano con el año de data_pedido (pista: .dt.year)
# 3. Crea la columna mes con el formato "2025-11" (pista: .dt.to_period("M").astype(str))
# 4. Muestra data_pedido, ano, mes, quantidade, preco_unitario y valor_total de las primeras filas

# ✍️ Tu código aquí:


### Paso 6 — Unir las tablas (merge)

Une pedidos con clientes para saber, en cada pedido, de qué estado y canal viene el cliente.

In [ ]:
# PASO 6 — UNIR LAS TABLAS (MERGE)
# 1. Une pedidos con clientes por la columna cliente_id: pedidos.merge(clientes, on=..., how="left")
#    Guarda el resultado en df. ("left" = mantener todos los pedidos)
# 2. Verifica que df tiene la MISMA cantidad de filas que pedidos (si aumenta, hay llaves duplicadas)
# 3. Crea vendas = solo los pedidos con status "Entregue" (cancelados y devueltos no son ingreso)
#    💡 Pista: df[df["status"] == "Entregue"].copy()

# ✍️ Tu código aquí:


### Paso 7 — Calcular los KPIs principales

Los **KPIs** (indicadores clave) son los números grandes que van arriba del dashboard.

In [ ]:
# PASO 7 — CALCULAR LOS KPIS PRINCIPALES
# 1. Crea una función brl(valor) que formatee 1234.5 como "R$ 1.234,50"
#    💡 Pista: f"{valor:,.2f}" da "1,234.50"; después intercambia comas y puntos
# 2. Calcula, usando vendas:
#    - faturamento = suma de valor_total
#    - n_pedidos = cantidad de pedido_id únicos (.nunique())
#    - ticket_medio = faturamento / n_pedidos
#    - clientes_compradores = cantidad de cliente_id únicos
# 3. Usando df (todos los pedidos): taxa_problema = % de pedidos con status distinto de "Entregue"
# 4. nota_media = promedio de avaliacao (el promedio ignora los vacíos automáticamente)
# 5. Imprime todo con print(f"...") de forma bonita

# ✍️ Tu código aquí:


### Paso 8 — Analítica por grupos (groupby)

Ahora responde las preguntas de negocio agrupando los datos: por mes, categoría, producto, estado y canal.

In [ ]:
# PASO 8 — ANALÍTICA POR GRUPOS (GROUPBY)
# 8.1 Facturación por mes: vendas.groupby("mes", as_index=False)["valor_total"].sum() → fat_mensal
# 8.2 Facturación por año y crecimiento 2025 vs 2024 en %: (fat_2025 / fat_2024) - 1
# 8.3 Facturación por categoría, ordenada de mayor a menor (.sort_values) → por_categoria
#     + columna participacao = valor de la categoría / total
#     + top 10 productos por facturación (pista: .nlargest(10))
# 8.4 Por estado: facturación y cantidad de clientes únicos → por_estado
#     💡 Pista: .agg(faturamento=("valor_total", "sum"), clientes=("cliente_id", "nunique"))
# 8.5 Por canal_aquisicao: clientes únicos, pedidos únicos y facturación → por_canal
#     + crea pedidos_por_cliente y valor_por_cliente. ¿Qué canal trae clientes más valiosos?
# 8.6 % de cada status por categoría: pd.crosstab(df["categoria"], df["status"], normalize="index")

# ✍️ Tu código aquí:


### Paso 9 — Segmentación de clientes RFM

**RFM** es una técnica clásica de marketing: **R**ecencia (¿hace cuántos días compró?), **F**recuencia (¿cuántas veces?) y **M**onto o valor (¿cuánto gastó?).

In [ ]:
# PASO 9 — SEGMENTACIÓN DE CLIENTES RFM
# 1. Define data_ref = fecha máxima de vendas + 1 día (el "hoy" del análisis)
# 2. Agrupa vendas por cliente_id y calcula:
#    - ultima_compra = max de data_pedido
#    - frequencia = cantidad de pedidos únicos
#    - valor = suma de valor_total
#    Guarda en rfm
# 3. recencia = (data_ref - ultima_compra) en días (pista: .dt.days)
# 4. Da notas de 1 a 5 por quintiles con pd.qcut(columna, 5, labels=[...]):
#    - R: recencia baja es MEJOR → labels=[5, 4, 3, 2, 1]
#    - F y M: valor alto es mejor → labels=[1, 2, 3, 4, 5]
#    💡 Usa columna.rank(method="first") dentro de qcut para evitar errores por empates
# 5. Escribe una función segmentar(r, f) que devuelva:
#    "Campeões" (R>=4 y F>=4), "Leais" (R>=3 y F>=3), "Novos" (R>=4),
#    "Em risco" (R<=2 y F>=3), "Hibernando" (R<=2), y si no "Precisam de atenção"
# 6. Aplica con rfm.apply(lambda linha: segmentar(linha["R"], linha["F"]), axis=1)
# 7. Resume por segmento: cantidad de clientes, recencia media, compras medias y valor medio

# ✍️ Tu código aquí:


### Paso 10 — Armar el dashboard y exportarlo

Junta todo en **un solo panel**: KPIs arriba (lo más importante primero), tendencia en el medio y los detalles abajo. Después lo guardas como HTML para abrir en cualquier navegador.

In [ ]:
# PASO 10 — ARMAR EL DASHBOARD Y EXPORTARLO
# 1. Crea la estructura con make_subplots(rows=4, cols=4, specs=[...]):
#    - fila 1: 4 indicadores (type "indicator") → Faturamento, Pedidos, Ticket médio, Clientes
#    - fila 2: un gráfico de línea que ocupa las 4 columnas (colspan=4) → facturación mensual
#    - fila 3: dos barras horizontales (colspan=2 cada una) → por categoría | valor por cliente por canal
#    - fila 4: dos barras → por estado | clientes por segmento RFM
# 2. Agrega cada gráfico con fig.add_trace(go.Indicator / go.Scatter / go.Bar, row=..., col=...)
# 3. Usa UN color para todas las barras y resalta en otro color solo lo que exige acción ("Em risco")
# 4. Marca el pico de ventas con fig.add_annotation
# 5. Ajusta el diseño con fig.update_layout (título, altura, separators=",." para formato brasileño)
# 6. Muestra con fig.show() y guarda con fig.write_html("dashboard_loja_verde.html")

# ✍️ Tu código aquí:


---
# 🔑 Parte B — Solución comentada

**Intenta resolver la Parte A antes de mirar esto.** Después compara: no hay una única forma correcta,
pero fíjate en las funciones que usé y en los comentarios que explican el porqué.

> Ejecuta la Parte B **en orden, desde el Paso 1**: vuelve a leer los archivos originales, así que no depende de lo que hiciste en la Parte A.

### ✅ Paso 1 — Importar las librerías

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.float_format", "{:,.2f}".format)  # muestra los números con 2 decimales

### ✅ Paso 2 — Importar las bases de datos

In [ ]:
clientes = pd.read_csv("clientes.csv")
pedidos = pd.read_csv("pedidos.csv", sep=";")  # ";" es común en archivos exportados en Brasil (Excel en portugués)

print("clientes:", clientes.shape)  # (filas, columnas)
print("pedidos: ", pedidos.shape)
display(pedidos.head())

### ✅ Paso 3 — Explorar y diagnosticar los problemas

In [ ]:
pedidos.info()  # preco_unitario y data_pedido aparecen como texto (object/str) → hay que convertir

print("\nVacíos por columna:")
print(pedidos.isna().sum())  # avaliacao tiene vacíos: es normal (no todos evalúan)

print("\nFilas duplicadas:", pedidos.duplicated().sum())
print("Cantidades <= 0:", (pedidos["quantidade"] <= 0).sum())

print("\nValores de estado (fíjate en las variaciones):")
print(clientes["estado"].value_counts())

display(clientes.describe())

💡 **Qué observar:** encontraste 4 problemas típicos de datos reales: (1) números guardados como texto ("R\$ 39,90"), (2) fechas como texto, (3) duplicados y errores de digitación, (4) categorías escritas de formas distintas ("SP", " sp"). Si no los arreglas, cada cálculo posterior sale mal.

### ✅ Paso 4 — Limpiar y ajustar el formato de los datos

In [ ]:
# 4.1 Duplicados
antes = len(pedidos)
pedidos = pedidos.drop_duplicates()
print(f"Duplicados eliminados: {antes - len(pedidos)}")

# 4.2 Precio: "R$ 1.234,56" → 1234.56  (el orden de los replace importa)
pedidos["preco_unitario"] = (
    pedidos["preco_unitario"]
    .str.replace("R$", "", regex=False)
    .str.replace(".", "", regex=False)   # 1º quita el punto de los miles
    .str.replace(",", ".", regex=False)  # 2º coma decimal → punto
    .str.strip()
    .astype(float)
)

# 4.3 Fechas: texto → datetime
pedidos["data_pedido"] = pd.to_datetime(pedidos["data_pedido"], format="%d/%m/%Y")
clientes["data_cadastro"] = pd.to_datetime(clientes["data_cadastro"])

# 4.4 Estado: sin espacios y en mayúsculas
clientes["estado"] = clientes["estado"].str.strip().str.upper()

# 4.5 Cantidades inválidas (.copy() evita avisos al modificar el resultado después)
pedidos = pedidos[pedidos["quantidade"] > 0].copy()

# 4.6 Edad vacía → mediana (la mediana no se deja influir por valores extremos)
clientes["idade"] = clientes["idade"].fillna(clientes["idade"].median())

# 4.7 Verificación
pedidos.info()
print(clientes["estado"].value_counts())

### ✅ Paso 5 — Crear columnas nuevas

In [ ]:
pedidos["valor_total"] = pedidos["quantidade"] * pedidos["preco_unitario"]
pedidos["ano"] = pedidos["data_pedido"].dt.year
pedidos["mes"] = pedidos["data_pedido"].dt.to_period("M").astype(str)  # ej.: "2025-11"

pedidos[["data_pedido", "ano", "mes", "quantidade", "preco_unitario", "valor_total"]].head()

### ✅ Paso 6 — Unir las tablas (merge)

In [ ]:
df = pedidos.merge(clientes, on="cliente_id", how="left")
print("pedidos:", len(pedidos), "| df:", len(df))  # deben ser iguales

# Ingreso real = solo pedidos entregados
vendas = df[df["status"] == "Entregue"].copy()
print(f"Pedidos entregados: {len(vendas)} de {len(df)}")

### ✅ Paso 7 — Calcular los KPIs principales

In [ ]:
def brl(valor):
    """Formatea un número como moneda brasileña: 1234.5 → 'R$ 1.234,50'."""
    return "R$ " + f"{valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

faturamento = vendas["valor_total"].sum()
n_pedidos = vendas["pedido_id"].nunique()
ticket_medio = faturamento / n_pedidos
clientes_compradores = vendas["cliente_id"].nunique()
taxa_problema = (df["status"] != "Entregue").mean()  # media de True/False = proporción
nota_media = vendas["avaliacao"].mean()

print("📊 KPIs Loja Verde (2024–2025)")
print(f"Faturamento total:       {brl(faturamento)}")
print(f"Pedidos entregados:      {n_pedidos}")
print(f"Ticket médio:            {brl(ticket_medio)}")
print(f"Clientes compradores:    {clientes_compradores}")
print(f"Cancelados + devolvidos: {taxa_problema:.1%}")
print(f"Nota média:              {nota_media:.2f} ⭐")

### ✅ Paso 8 — Analítica por grupos (groupby)

In [ ]:
# 8.1 Evolución mensual
fat_mensal = vendas.groupby("mes", as_index=False)["valor_total"].sum()
display(fat_mensal.tail(6))

# 8.2 Crecimiento anual
fat_ano = vendas.groupby("ano")["valor_total"].sum()
crescimento = fat_ano[2025] / fat_ano[2024] - 1
print(f"Facturación 2024: {brl(fat_ano[2024])} | 2025: {brl(fat_ano[2025])} | crecimiento: {crescimento:.1%}")

# 8.3 Categorías y top 10 productos
por_categoria = (vendas.groupby("categoria", as_index=False)["valor_total"].sum()
                 .sort_values("valor_total", ascending=False))
por_categoria["participacao"] = por_categoria["valor_total"] / por_categoria["valor_total"].sum()
display(por_categoria)
display(vendas.groupby("produto")["valor_total"].sum().nlargest(10))

# 8.4 Estados
por_estado = (vendas.groupby("estado")
              .agg(faturamento=("valor_total", "sum"), clientes=("cliente_id", "nunique"))
              .sort_values("faturamento", ascending=False))
display(por_estado)

# 8.5 Canal de adquisición: ¿quién trae los clientes más valiosos?
por_canal = vendas.groupby("canal_aquisicao").agg(
    clientes=("cliente_id", "nunique"),
    pedidos=("pedido_id", "nunique"),
    faturamento=("valor_total", "sum"),
)
por_canal["pedidos_por_cliente"] = por_canal["pedidos"] / por_canal["clientes"]
por_canal["valor_por_cliente"] = por_canal["faturamento"] / por_canal["clientes"]
display(por_canal.sort_values("valor_por_cliente", ascending=False))

# 8.6 Cancelados/devueltos por categoría, en %
display(pd.crosstab(df["categoria"], df["status"], normalize="index").mul(100).round(1))

💡 **Qué observar:** el canal con **más clientes** (Instagram) no es el que trae **los clientes más valiosos**. Los clientes que llegan por **Indicação** compran más veces y dejan más dinero cada uno. Esa es la clase de insight que cambia una decisión de marketing: tal vez convenga un programa de "indica y gana".

### ✅ Paso 9 — Segmentación de clientes RFM

In [ ]:
data_ref = vendas["data_pedido"].max() + pd.Timedelta(days=1)  # "hoy" del análisis

rfm = vendas.groupby("cliente_id").agg(
    ultima_compra=("data_pedido", "max"),
    frequencia=("pedido_id", "nunique"),
    valor=("valor_total", "sum"),
)
rfm["recencia"] = (data_ref - rfm["ultima_compra"]).dt.days

# Notas 1–5 por quintiles (5 = mejor). rank(method="first") desempata para que los cortes no se repitan.
rfm["R"] = pd.qcut(rfm["recencia"].rank(method="first"), 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["F"] = pd.qcut(rfm["frequencia"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M"] = pd.qcut(rfm["valor"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)

def segmentar(r, f):
    if r >= 4 and f >= 4:
        return "Campeões"
    if r >= 3 and f >= 3:
        return "Leais"
    if r >= 4:
        return "Novos"
    if r <= 2 and f >= 3:
        return "Em risco"
    if r <= 2:
        return "Hibernando"
    return "Precisam de atenção"

rfm["segmento"] = rfm.apply(lambda linha: segmentar(linha["R"], linha["F"]), axis=1)

resumo_rfm = (rfm.groupby("segmento")
              .agg(clientes=("R", "size"), recencia_media=("recencia", "mean"),
                   compras_medias=("frequencia", "mean"), valor_medio=("valor", "mean"))
              .sort_values("valor_medio", ascending=False))
display(resumo_rfm)

💡 **Qué hacer con cada segmento:** **Campeões** → programa VIP y pedirles que indiquen amigos. **Leais** → ofertas de productos complementarios. **Novos** → secuencia de bienvenida para la 2ª compra. **Em risco** → ¡prioridad! Eran buenos clientes y dejaron de comprar: cupón de "te extrañamos". **Hibernando** → campaña barata o dejar ir.

### ✅ Paso 10 — Armar el dashboard y exportarlo

In [ ]:
AZUL, LARANJA = "#2a78d6", "#eb6834"  # un color para todo + uno para destacar lo que exige acción

fig = make_subplots(
    rows=4, cols=4,
    specs=[
        [{"type": "indicator"}] * 4,
        [{"colspan": 4}, None, None, None],
        [{"colspan": 2}, None, {"colspan": 2}, None],
        [{"colspan": 2}, None, {"colspan": 2}, None],
    ],
    row_heights=[0.07, 0.31, 0.31, 0.31],
    vertical_spacing=0.08, horizontal_spacing=0.16,
    subplot_titles=["", "", "", "",
                    "Facturación mensual (R$)",
                    "Facturación por categoría (R$)", "Valor medio por cliente, por canal (R$)",
                    "Facturación por estado (R$)", "Clientes por segmento RFM"],
)

# Fila 1 — KPIs
kpis = [("Faturamento", faturamento, "R$ ", ",.0f"), ("Pedidos", n_pedidos, "", ",.0f"),
        ("Ticket médio", ticket_medio, "R$ ", ",.2f"), ("Clientes", clientes_compradores, "", ",.0f")]
for i, (titulo, valor, prefixo, formato) in enumerate(kpis, start=1):
    fig.add_trace(go.Indicator(mode="number", value=valor, title={"text": titulo, "font": {"size": 15}},
                               number={"prefix": prefixo, "valueformat": formato, "font": {"size": 34}}),
                  row=1, col=i)

# Fila 2 — tendencia mensual + anotación del pico
fig.add_trace(go.Scatter(x=fat_mensal["mes"], y=fat_mensal["valor_total"], mode="lines+markers",
                         line=dict(color=AZUL, width=2), marker=dict(size=8),
                         hovertemplate="%{x}<br>R$ %{y:,.2f}<extra></extra>"), row=2, col=1)
pico = fat_mensal.loc[fat_mensal["valor_total"].idxmax()]
fig.add_annotation(x=pico["mes"], y=pico["valor_total"], text=f"Pico: {pico['mes']} (Black Friday)",
                   showarrow=True, arrowhead=0, ax=-70, ay=-25, row=2, col=1)

# Fila 3 — categorías | canales
cat = por_categoria.sort_values("valor_total")
fig.add_trace(go.Bar(x=cat["valor_total"], y=cat["categoria"], orientation="h", marker_color=AZUL,
                     hovertemplate="%{y}<br>R$ %{x:,.2f}<extra></extra>"), row=3, col=1)
canal = por_canal.sort_values("valor_por_cliente")
fig.add_trace(go.Bar(x=canal["valor_por_cliente"], y=canal.index, orientation="h", marker_color=AZUL,
                     hovertemplate="%{y}<br>R$ %{x:,.2f} por cliente<extra></extra>"), row=3, col=3)

# Fila 4 — estados | segmentos RFM (resaltamos "Em risco")
fig.add_trace(go.Bar(x=por_estado.index, y=por_estado["faturamento"], marker_color=AZUL,
                     hovertemplate="%{x}<br>R$ %{y:,.2f}<extra></extra>"), row=4, col=1)
seg = resumo_rfm.sort_values("clientes")
cores = [LARANJA if s == "Em risco" else AZUL for s in seg.index]
fig.add_trace(go.Bar(x=seg["clientes"], y=seg.index, orientation="h", marker_color=cores,
                     hovertemplate="%{y}: %{x} clientes<extra></extra>"), row=4, col=3)

fig.update_traces(marker_cornerradius=4, selector=dict(type="bar"))
fig.update_layout(
    title=dict(text="<b>Loja Verde</b> · Dashboard de ventas 2024–2025", x=0.01, font=dict(size=22)),
    template="plotly_white", height=1250, showlegend=False,
    separators=",.",  # formato brasileño: 1.234,56
    font=dict(family="system-ui, -apple-system, Segoe UI, sans-serif", size=13, color="#0b0b0b"),
    paper_bgcolor="#fcfcfb", plot_bgcolor="#fcfcfb", bargap=0.35,
    margin=dict(t=100, l=40, r=30, b=40),
)
fig.update_xaxes(gridcolor="#e1e0d9", linecolor="#c3c2b7", zeroline=False)
fig.update_yaxes(gridcolor="#e1e0d9", linecolor="#c3c2b7", zeroline=False)
fig.show()

fig.write_html("dashboard_loja_verde.html", include_plotlyjs="cdn")
print("✅ Dashboard guardado: dashboard_loja_verde.html (en Colab: panel 📁 → ⋮ → Descargar)")

💡 **Buenas prácticas de dashboard que aplicaste:** lo más importante arriba (KPIs); un color para todo y un segundo color solo para lo que exige acción; números en formato local (R\$ 1.234,56); títulos que dicen qué se mide y en qué unidad; nada de gráficos de torta 3D ni dos ejes Y en el mismo gráfico.

---
## 🎤 Cómo presentarlo a Marina (5 slides)
1. **Contexto:** qué datos usaste y qué limpiaste (duplicados, precios en texto, estados sucios).
2. **Números grandes:** facturación, ticket medio, crecimiento 2025 vs 2024.
3. **Insight 1:** la estacionalidad (Black Friday y Navidad concentran ventas → planificar stock).
4. **Insight 2:** Indicação trae menos clientes, pero los más valiosos → programa de referidos.
5. **Acción:** recuperar a los clientes "Em risco" con una campaña de reactivación.

## 🚀 Retos extra
1. Agrega la **M** (valor) en la segmentación RFM. ¿Cambian los segmentos?
2. Crea franjas de edad con `pd.cut(clientes["idade"], bins=[17, 25, 35, 45, 60, 80])` y descubre qué categoría prefiere cada franja (`pd.crosstab`).
3. **Cohortes:** agrupa clientes por mes de registro y calcula qué % sigue comprando 3, 6 y 12 meses después.
4. Compara el ticket medio de noviembre (con descuento) contra el resto del año. ¿El descuento se paga con más volumen?
5. **Nivel pro (en computadora):** convierte el dashboard en una app web con **Streamlit** (`pip install streamlit`, guarda el código en `app.py` y ejecuta `streamlit run app.py`), con un filtro de año y de estado.